In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Ffull_frame%2Futility_pole_full_frame_analysis.ipynb?utm_source=full_frame_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Full-Frame Utility pole analysis with Gemini 3.5 Flash

This notebook demonstrates how to analyze full-frame images of utility poles from Imagery Insights using the Gemini 3.5 Flash model via Vertex AI, overlaying BigQuery-reported bounding boxes, and calculating individual API costs.

## Install Required Libraries

In [ ]:
!pip install --upgrade google-cloud-bigquery google-genai google-cloud-storage "pillow<11.0.0" matplotlib

## Configuration

**Important**: Replace the placeholder values below with your actual GCP Project ID and Region.

In [ ]:
PROJECT_ID = 'YOUR_PROJECT_ID'  # @param {type:"string"}
REGION = 'global'      # @param {type:"string"}

# BigQuery Configuration
BIGQUERY_DATASET_ID = 'imagery_insights___us' # @param {type:"string"}
BIGQUERY_TABLE_ID = 'full_frame_observations_latest' # @param {type:"string"}
QUERY_LIMIT = 10 # @param {type:"integer"}
ASSET_TYPE = "ASSET_CLASS_UTILITY_POLE" # @param {type:"string"}
MODEL = "gemini-3.5-flash" # @param {type:"string"}
THINKING_LEVEL = "HIGH" # @param ["MINIMAL", "LOW", "MEDIUM", "HIGH"] {type:"string"}

## Imports and SDK Initialization

In [ ]:
import io
import vertexai
import PIL.Image
import PIL.ImageDraw
import matplotlib.pyplot as plt
from google.cloud import bigquery
from google.cloud import storage
from google import genai
from google.genai import types
from google.genai.types import Content, Part

# Initialize Vertex AI SDK and Gemini Client
vertexai.init(project=PROJECT_ID, location=REGION)
client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

## Fetch Image URIs and Bounding Boxes from BigQuery

We query the BigQuery table to get the GCS URIs of the images and their bounding box coordinates.

In [ ]:
BIGQUERY_SQL_QUERY = f"""
SELECT
  *
FROM
  `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.{BIGQUERY_TABLE_ID}`
WHERE asset_type = '{ASSET_TYPE}'
LIMIT {QUERY_LIMIT};
"""

# Execute BigQuery Query
try:
    bigquery_client = bigquery.Client(project=PROJECT_ID)
    query_job = bigquery_client.query(BIGQUERY_SQL_QUERY)
    query_response_data = [dict(row) for row in query_job]
    
    observations = []
    for item in query_response_data:
        if item.get("gcs_uri") and item.get("bbox"):
            observations.append({
                "gcs_uri": item.get("gcs_uri"),
                "bbox": item.get("bbox"),
                "asset_type": item.get("asset_type")
            })

    print(f"Successfully fetched {len(observations)} observations.")
    for obs in observations:
        print(obs['gcs_uri'])
except Exception as e:
    print(f"An error occurred while querying BigQuery: {e}")

## Bounding Box Visualization Helper

Define a function to load the image from GCS, draw the reported bounding box, and render the output image inside the notebook.

In [ ]:
def display_image_with_bbox(gcs_uri: str, bbox: dict, label: str = None):
    """
    Downloads the image from GCS, draws the bounding box, and displays it.
    """
    try:
        if not gcs_uri.startswith("gs://"):
            print("Invalid GCS URI")
            return
            
        parts = gcs_uri[5:].split("/", 1)
        bucket_name = parts[0]
        blob_name = parts[1]
        
        storage_client = storage.Client(project=PROJECT_ID)
        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        image_bytes = blob.download_as_bytes()
        
        image = PIL.Image.open(io.BytesIO(image_bytes))
        draw = PIL.ImageDraw.Draw(image)
        
        # BQ bbox coordinates
        xmin = bbox['lo']['x']
        ymin = bbox['lo']['y']
        xmax = bbox['hi']['x']
        ymax = bbox['hi']['y']
        
        # Draw bounding box
        draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=10)
        
        if label:
            # Simple text overlay
            draw.text((xmin + 20, ymin + 20), label, fill="red")
            
        plt.figure(figsize=(12, 8))
        plt.imshow(image)
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Error displaying image: {e}")

## Define Image Classification Function

This function submits the image to Gemini along with our prompt, retrieves the response, and calculates the API call cost based on usage metadata.

In [ ]:
def classify_image_with_gemini(gcs_uri: str, prompt: str) -> tuple[str, float, int, int]:
    """
    Classifies an image using the Gemini model by directly passing its GCS URI.
    Returns the classification text, calculated cost, prompt token count, and candidates token count.
    """
    try:
        contents = [
            prompt,
            Part(file_data={'file_uri': gcs_uri, 'mime_type': 'image/jpeg'})
        ]
        
        # Configure Gemini generation config with the desired thinking budget/level
        config = types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                thinking_level=THINKING_LEVEL
            )
        )
        
        response = client.models.generate_content(model=MODEL, contents=contents, config=config)
        
        # Calculate cost dynamically
        prompt_tokens = response.usage_metadata.prompt_token_count
        completion_tokens = response.usage_metadata.candidates_token_count
        
        # Pricing for gemini-3.5-flash: Input: $0.000075 / 1k, Output: $0.00030 / 1k
        input_cost = prompt_tokens * (0.000075 / 1000)
        output_cost = completion_tokens * (0.00030 / 1000)
        total_cost = input_cost + output_cost
        
        return response.text, total_cost, prompt_tokens, completion_tokens
    except Exception as e:
        print(f"Error classifying image from URI {gcs_uri}: {e}")
        return "Classification failed.", 0.0, 0, 0

## Run Analysis on Observations

Finally, loop through all observations, rendering each image with its bounding box, running the classification prompt, and outputting both the prediction results and the cost of the API invocation.

In [ ]:
prompt = """You will be provided with a photo of a utility pole:
{photo_of_utility_pole}

Instructions:

1. Analyze the provided image. If the image does not clearly show a utility pole, return: {\"error\": \"No utility pole detected in the image.\"}
2. Detect and count the following:
    * Transformers
    * Power lines coming from the pole
    * Street lamps attached to the pole
    * Telephone or junction boxes
3. Assess the overall condition of the pole. Look for visible damage, bird nests, or other issues.  If the pole appears to be in good condition, note \"OK\".
4. Note the material with which the pole is made
5. Determine primary type of pole: Report this in the type field:
  * Street light
  * High tension power transmission
  * electricity pole
  * other
6. Provide your findings in the following JSON format:

```json
{
  \"pole_condition\": \"OK/Damaged/Other Issues\",
  \"type\": <pole_type>,
   \"material\":<material>
  \"transformers\": <number_of_transformers>,
  \"power_lines\": <number_of_power_lines>,
  \"street_lamps\": <number_of_street_lamps>,
  \"junction_boxes\": <number_of_junction_boxes>,
  \"additional_notes\": \"<any_other_observations>\"
}
```

Example:

Let's say the image shows a utility pole in good condition with one transformer, three power lines, one street lamp, and no junction boxes. The JSON output would be:

```json
{
  \"pole_condition\": \"OK\",
  \"type\": electricity pole,
  \"material\": wood,
  \"transformers\": 1,
  \"power_lines\": 3,
  \"street_lamps\": 1,
  \"junction_boxes\": 0,
  \"additional_notes\": \"None\"
}
```
"""

if 'observations' in locals() and observations:
    for obs in observations:
        uri = obs['gcs_uri']
        bbox = obs['bbox']
        asset_type = obs['asset_type']
        
        print(f"\n==================================================")
        print(f"Analyzing {uri}")
        print(f"==================================================")
        
        # Display image with bounding box
        print("Drawing BigQuery reported bounding box...")
        display_image_with_bbox(uri, bbox, label=asset_type)
        
        # Classify image
        print("Running Gemini classification...")
        classification, cost, in_tokens, out_tokens = classify_image_with_gemini(uri, prompt)
        print(f"Result:\n{classification}")
        print(f"Tokens - Input: {in_tokens} | Output: {out_tokens}")
        print(f"Gemini API cost for this image: ${cost:.6f}")
else:
    print("No observations were found to analyze.")